In [1]:
# # Environment Testing :
# from langchain_openai import ChatOpenAI
# import os 
# from dotenv import load_dotenv

# load_dotenv()

# GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

# llm = ChatOpenAI(
#     model="gpt-4.1-mini",
#     base_url="https://models.github.ai/inference",
#     temperature=0.1,
#     max_tokens=500,
#     api_key=GITHUB_TOKEN
# )

# response = llm.invoke("What is the capital of India?")
# print(response.content)

### Read PDF Files:

In [2]:
# Load Libraries:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os 

# Read All Pdf's :
def process_pdf(pdf_direcory):
    
    all_documents = []
    pdf_dir = Path(pdf_direcory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf process")
    
    for p_file in pdf_files:
        print(f"\nProcess : {p_file}")

        try:
            loader = PyMuPDFLoader(str(p_file))
            documents = loader.load()

            # Add Source Info for metadata:
            for doc in documents:
                doc.metadata['source_file'] = p_file.name
                doc.metadata['file_type'] = "pdf"

            all_documents.extend(documents)
            print(f"💯Loded {len(documents)} Pages")
        
        except Exception as e:
            print(f"✖️ Error {e}")
        
    print(f"Total Documents Loded: {len(all_documents)}")
    return all_documents



all_pdf_documents = process_pdf(r"D:\Code\AI-ML\Conversational_RAG\Data")
# all_pdf_documents = process_pdf(r"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\Data")
# all_pdf_documents = process_pdf(r"D:\Code\AI-ML\Conversational_RAG\Vectore_DB")

Found 4 pdf process

Process : D:\Code\AI-ML\Conversational_RAG\Data\Fundamental of Data Science.pdf
💯Loded 8 Pages

Process : D:\Code\AI-ML\Conversational_RAG\Data\Introduction to Outlier.docx.pdf
💯Loded 22 Pages

Process : D:\Code\AI-ML\Conversational_RAG\Data\NIST-CSF-NOTES.pdf
💯Loded 21 Pages

Process : D:\Code\AI-ML\Conversational_RAG\Data\NIST-CyberSecurityFramework.pdf
💯Loded 32 Pages
Total Documents Loded: 83


### Chunking Documnets:

In [3]:
# Text Splitting get into chunks
def split_documnets(documnets, chunk_size=1000, overlap = 200):
    """Split Documents into Chunks."""
    text_chunk = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = overlap,
        length_function = len,
        separators= ["\n\n","\n"," ", ""] #\n\n is a Peragraph Seprator
    )

    split_doc = text_chunk.split_documents(documnets)
    print(f"Split {len(documnets)} Documnets into {len(split_doc)} chunks.")

    if split_doc:
        print(f"\nExample Chunk:\nContent : \n{split_doc[0].page_content[:200]}...\nMetaData: {split_doc[0].metadata}")
    
    return split_doc

chunks = split_documnets(all_pdf_documents)


Split 83 Documnets into 185 chunks.

Example Chunk:
Content : 
-: Fundamental of Data Science :- 
 
Q.1 :- Explain Data-Science with proper Diagram. 
➢ Ans:- Data Science : 
- Data Science is a process to perform tasks on a Row-Data / Row-Things and manage the da...
MetaData: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-07-13T18:08:17+05:30', 'source': 'D:\\Code\\AI-ML\\Conversational_RAG\\Data\\Fundamental of Data Science.pdf', 'file_path': 'D:\\Code\\AI-ML\\Conversational_RAG\\Data\\Fundamental of Data Science.pdf', 'total_pages': 8, 'format': 'PDF 1.7', 'title': '', 'author': 'Karan Mistry', 'subject': '', 'keywords': '', 'moddate': '2025-07-13T18:08:17+05:30', 'trapped': '', 'modDate': "D:20250713180817+05'30'", 'creationDate': "D:20250713180817+05'30'", 'page': 0, 'source_file': 'Fundamental of Data Science.pdf', 'file_type': 'pdf'}


### vectore Store:

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
import os

def vector_store(chunks, save_path="faiss_index"):
    
    print("\n🔄 Creating Embeddings (FAISS)...")

    embedding_model = SentenceTransformerEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    vector_db = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    # Ensure directory exists
    os.makedirs(save_path, exist_ok=True)

    # Save locally
    vector_db.save_local(save_path)

    print(f"✅ FAISS Vector DB Created with {len(chunks)} chunks")
    print(f"📁 Saved at: {save_path}")

    return vector_db

vectore_db = vector_store(chunks,"D:\Code\AI-ML\Conversational_RAG\Vectore_DB")
# vectore_db = vector_store(chunks,"D:\Code\Python\Intership_Learning\AI_Ml_Projects\04_Conversational_RAG\Vectore_DB")


🔄 Creating Embeddings (FAISS)...


<>:29: SyntaxWarning: invalid escape sequence '\C'
<>:29: SyntaxWarning: invalid escape sequence '\C'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_26728\3834290262.py:29: SyntaxWarning: invalid escape sequence '\C'
  vectore_db = vector_store(chunks,"D:\Code\AI-ML\Conversational_RAG\Vectore_DB")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_26728\3834290262.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS Vector DB Created with 185 chunks
📁 Saved at: D:\Code\AI-ML\Conversational_RAG\Vectore_DB


### Chat with LLM:

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
import os

# ✅ Shared embedding model (same as PDF)
embedding_model = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# ✅ User knowledge vector DB (EMPTY INIT)
user_knowledge_db = FAISS.from_texts(
    texts=[""],   # dummy init (FAISS needs at least 1)
    embedding=embedding_model
)


# 🔥 Query Rewriting (keep this)
def rewrite_query(llm, chat_history, query):
    prompt = f"""
Rewrite the question to be standalone.

CHAT HISTORY:
{chat_history}

QUESTION:
{query}

REWRITTEN QUESTION:
"""
    response = llm.invoke(prompt)
    return response.content.strip()


def start_chat(vector_db):
    """Conversational RAG with dynamic user learning"""

    print("\n🤖 Chatbot Started (type 'exit' to stop)\n")

    # ✅ LLM
    llm = ChatOpenAI(
        model="gpt-4.1-mini",
        base_url="https://models.github.ai/inference",
        temperature=0.1,
        max_tokens=500,
        api_key=os.getenv("GITHUB_TOKEN")
    )

    # ✅ Memory (for rewriting only)
    memory = ChatMessageHistory()

    # ✅ PDF retriever
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    while True:
        query = input("User: ")

        if query.lower() == "exit":
            print("\n👋 Exiting chatbot...")
            break

        # 🧠 Build chat history (limited)
        chat_history = "\n".join([
            f"User: {msg.content}" if msg.type == "human" else f"Bot: {msg.content}"
            for msg in memory.messages[-4:]
        ])

        # 🔥 Rewrite query
        if chat_history:
            rewritten_query = rewrite_query(llm, chat_history, query)
        else:
            rewritten_query = query

        # 🔍 Retrieve from PDF
        doc_results = retriever.invoke(rewritten_query)

        # 🔍 Retrieve from USER knowledge
        try:
            user_results = user_knowledge_db.similarity_search(rewritten_query, k=2)
        except:
            user_results = []

        # 🔥 Combine both
        docs = doc_results + user_results

        if not docs:
            answer = "I don't know based on the provided documents."
        else:
            # 📄 Build context
            context = "\n\n".join([doc.page_content for doc in docs if doc.page_content.strip()])

            # ✅ Final prompt (both sources included)
            final_prompt = f"""
            Answer using the context below.

            Context may include:
            - Document knowledge
            - User-provided knowledge from conversation

            If answer is not found, say:
            "I don't know based on the provided documents."

            CONTEXT:
            {context}

            QUESTION:
            {query}

            ANSWER:
            """
            response = llm.invoke(final_prompt)
            answer = response.content.strip()

        # 💾 Save conversation memory
        memory.add_user_message(query)
        memory.add_ai_message(answer)

        # 🧠 Store user input as knowledge (AFTER answering)
        if query.strip():
            user_knowledge_db.add_texts([query])

        # ✅ Clean output
        print(f"\n👤User: {query}")
        print(f"🤖Bot: {answer}\n")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# ✅ RUN
chat = start_chat(vectore_db)


🤖 Chatbot Started (type 'exit' to stop)


👤User: hyy my name is karan.
🤖Bot: Hello Karan! How can I assist you today?


👤User: which NIST version are mention in doc ? is that NIST Version 0.1 ?
🤖Bot: The document mentions the NIST Cybersecurity Framework (CSF) 2.0, dated February 26, 2024. It does not refer to NIST Version 0.1.


👤User: howmany programming language mention in doc fro Data science /
🤖Bot: The document mentions five programming languages for data science:

1. Python  
2. R  
3. SQL  
4. Java  

(Note: Python, R, and SQL are explicitly listed, and Java is also mentioned as used in large-scale systems and big data frameworks.)


👤User: what's my name ?
🤖Bot: Your name is Karan.


👋 Exiting chatbot...
